# Proyecto Final - Entregable 1  
## Análisis exploratorio, limpieza, correlación y modelos predictivos  
### Dataset: Bike Sharing Dataset - `hour.csv`

**Pregunta principal:**  
¿En qué horas del día se concentra la mayor demanda de bicicletas y cómo varía ese patrón entre días laborables y fines de semana?

**Subpreguntas:**
- ¿Cuáles son las horas pico de uso durante un día laboral vs un fin de semana?
- ¿El comportamiento horario de usuarios casuales difiere del de usuarios registrados?
- ¿Las condiciones climáticas afectan la demanda en ciertas horas más que en otras?
- ¿Hay franjas horarias con demanda consistentemente baja independientemente del día?

Este notebook desarrolla los procedimientos del Entregable 1: análisis preliminar, visualización, análisis de valores faltantes, eliminación/imputación, correlaciones, modelos de clasificación y modelos de regresión.

## 0. Instalación de librerías

Ejecuta esta celda solo si alguna librería no está instalada en tu entorno.

In [ ]:
%pip install pandas numpy matplotlib seaborn missingno upsetplot scikit-learn joblib

## 1. Importación de librerías

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from upsetplot import UpSet

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Carga del dataset

El proyecto trabajará con la tabla `hour.csv`, ya que contiene la demanda agregada por hora y permite responder la pregunta principal sobre los patrones horarios de uso.

In [ ]:
df = pd.read_csv("hour.csv")
df.head()

## 3. Descripción general del dataset

Según la documentación del dataset, `hour.csv` contiene registros de alquiler de bicicletas agregados por hora.  
Cada fila representa una hora y contiene variables temporales, climáticas y de demanda.

In [ ]:
# Cantidad de observaciones y variables
observaciones, variables = df.shape

print("Cantidad de observaciones:", observaciones)
print("Cantidad de variables:", variables)

In [ ]:
# Información general del dataset
df.info()

In [ ]:
# Tipos de datos por variable
tipos_datos = pd.DataFrame({
    "Variable": df.columns,
    "Tipo de dato": df.dtypes.astype(str).values,
    "Valores únicos": df.nunique().values
})

tipos_datos

## 4. Diccionario de variables principales

Variables principales del dataset:

- `instant`: índice del registro.
- `dteday`: fecha.
- `season`: estación del año.
- `yr`: año codificado.
- `mnth`: mes.
- `hr`: hora del día.
- `holiday`: indica si el día es feriado.
- `weekday`: día de la semana.
- `workingday`: indica si es día laboral.
- `weathersit`: condición climática.
- `temp`: temperatura normalizada.
- `atemp`: sensación térmica normalizada.
- `hum`: humedad normalizada.
- `windspeed`: velocidad del viento normalizada.
- `casual`: usuarios casuales.
- `registered`: usuarios registrados.
- `cnt`: total de bicicletas alquiladas.

## 5. Preparación inicial de variables para análisis

Se crean etiquetas más interpretables para facilitar las gráficas y análisis del proyecto.

In [ ]:
df_eda = df.copy()

# Conversión de fecha
df_eda["dteday"] = pd.to_datetime(df_eda["dteday"])

# Etiquetas para variables categóricas codificadas
map_season = {
    1: "Primavera",
    2: "Verano",
    3: "Otoño",
    4: "Invierno"
}

map_year = {
    0: "2011",
    1: "2012"
}

map_weekday = {
    0: "Domingo",
    1: "Lunes",
    2: "Martes",
    3: "Miércoles",
    4: "Jueves",
    5: "Viernes",
    6: "Sábado"
}

map_weather = {
    1: "Despejado / Parcialmente nublado",
    2: "Neblina / Nublado",
    3: "Lluvia o nieve ligera",
    4: "Lluvia fuerte / tormenta"
}

df_eda["season_label"] = df_eda["season"].map(map_season)
df_eda["yr_label"] = df_eda["yr"].map(map_year)
df_eda["weekday_label"] = df_eda["weekday"].map(map_weekday)
df_eda["weathersit_label"] = df_eda["weathersit"].map(map_weather)
df_eda["tipo_dia"] = np.where(df_eda["workingday"] == 1, "Día laboral", "Fin de semana o feriado")

# Variables climáticas en escalas interpretables
df_eda["temp_celsius_aprox"] = df_eda["temp"] * 41
df_eda["atemp_celsius_aprox"] = df_eda["atemp"] * 50
df_eda["hum_porcentaje"] = df_eda["hum"] * 100
df_eda["windspeed_aprox"] = df_eda["windspeed"] * 67

df_eda.head()

## 6. Estadísticas descriptivas

Se presentan medidas de tendencia central y dispersión de las variables numéricas.

In [ ]:
df_eda.describe().T

In [ ]:
# Estadísticas descriptivas para variables categóricas creadas
df_eda[["season_label", "yr_label", "weekday_label", "weathersit_label", "tipo_dia"]].describe()

## 7. Análisis de valores faltantes

El proyecto solicita analizar valores faltantes mediante tablas y gráficas como `missingno` y `UpSetPlot`.  
En este dataset se verificará si existen valores ausentes y, si no existen, se dejará documentado dentro del notebook.

In [ ]:
# Tabla resumen de valores faltantes
cantidad_faltantes = df_eda.isnull().sum()
porcentaje_faltantes = (df_eda.isnull().mean() * 100).round(2)

resumen_faltantes = pd.DataFrame({
    "Variable": df_eda.columns,
    "Valores faltantes": cantidad_faltantes.values,
    "Porcentaje faltante (%)": porcentaje_faltantes.values,
    "Tipo de dato": df_eda.dtypes.astype(str).values
}).sort_values(by="Valores faltantes", ascending=False)

resumen_faltantes

In [ ]:
# Variable con mayor ausencia
variable_mayor_ausencia = resumen_faltantes.iloc[0]["Variable"]
mayor_cantidad_faltantes = resumen_faltantes.iloc[0]["Valores faltantes"]
mayor_porcentaje_faltante = resumen_faltantes.iloc[0]["Porcentaje faltante (%)"]

print("Variable con mayor ausencia:", variable_mayor_ausencia)
print("Cantidad de valores faltantes:", mayor_cantidad_faltantes)
print("Porcentaje de ausencia:", mayor_porcentaje_faltante, "%")

### 7.1. Visualización con missingno

In [ ]:
# Matriz de valores faltantes con missingno.
# Aunque el dataset esté completo, esta gráfica sirve para evidenciar la revisión.

msno.matrix(df_eda)
plt.title("Matriz de valores faltantes - Bike Sharing hour.csv")
plt.show()

In [ ]:
# Gráfica de barras con missingno

msno.bar(df_eda)
plt.title("Valores presentes por variable - Bike Sharing hour.csv")
plt.show()

### 7.2. Visualización con UpSetPlot

UpSetPlot permite observar combinaciones de variables que faltan al mismo tiempo.  
Si el dataset no tiene valores faltantes, la gráfica no presenta intersecciones, por lo que se documenta el resultado.

In [ ]:
columnas_con_faltantes = df_eda.columns[df_eda.isnull().any()].tolist()

if len(columnas_con_faltantes) == 0:
    print("El dataset no presenta valores faltantes. No existen combinaciones de ausencia para graficar con UpSetPlot.")
else:
    columnas_top_faltantes = (
        df_eda[columnas_con_faltantes]
        .isnull()
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .index
        .tolist()
    )

    matriz_faltantes = df_eda[columnas_top_faltantes].isnull()
    datos_upset = matriz_faltantes.groupby(columnas_top_faltantes).size()
    datos_upset = datos_upset[datos_upset.index.map(lambda combinacion: any(combinacion))]

    UpSet(datos_upset, subset_size="count", show_counts=True).plot()
    plt.suptitle("Combinaciones de valores faltantes en el dataset")
    plt.show()

In [ ]:
# Tabla de grupos de valores faltantes

if len(columnas_con_faltantes) == 0:
    grupos_faltantes = pd.DataFrame(columns=["Variables ausentes en el grupo", "Cantidad de filas"])
    print("No hay grupos de valores faltantes porque el dataset no tiene datos ausentes.")
else:
    matriz_faltantes = df_eda[columnas_top_faltantes].isnull()
    grupos_faltantes = matriz_faltantes.groupby(columnas_top_faltantes).size().reset_index(name="Cantidad de filas")
    grupos_faltantes = grupos_faltantes[grupos_faltantes[columnas_top_faltantes].any(axis=1)]
    grupos_faltantes["Variables ausentes en el grupo"] = grupos_faltantes[columnas_top_faltantes].apply(
        lambda fila: ", ".join([columna for columna in columnas_top_faltantes if fila[columna]]),
        axis=1
    )
    grupos_faltantes = grupos_faltantes.sort_values(by="Cantidad de filas", ascending=False)
    display(grupos_faltantes[["Variables ausentes en el grupo", "Cantidad de filas"]])

## 8. Verificación de limpieza del dataset

Se revisan valores faltantes, duplicados, columnas completamente vacías y valores infinitos.

In [ ]:
total_faltantes = df_eda.isnull().sum().sum()
filas_duplicadas = df_eda.duplicated().sum()
columnas_completamente_vacias = df_eda.columns[df_eda.isnull().all()].tolist()

df_numerico = df_eda.select_dtypes(include=[np.number])
valores_infinitos = np.isinf(df_numerico).sum().sum()

resumen_limpieza = pd.DataFrame({
    "Aspecto evaluado": [
        "Cantidad de observaciones",
        "Cantidad de variables",
        "Total de valores faltantes",
        "Filas duplicadas",
        "Columnas completamente vacías",
        "Valores infinitos en columnas numéricas"
    ],
    "Resultado": [
        observaciones,
        variables,
        total_faltantes,
        filas_duplicadas,
        len(columnas_completamente_vacias),
        valores_infinitos
    ]
})

resumen_limpieza

In [ ]:
if total_faltantes == 0 and filas_duplicadas == 0 and len(columnas_completamente_vacias) == 0 and valores_infinitos == 0:
    print("Resultado: El dataset está limpio según los criterios evaluados.")
else:
    print("Resultado: El dataset requiere tratamiento antes de continuar.")

## 9. Eliminación o imputación de datos faltantes

La regla del proyecto indica considerar la eliminación de variables con más de 250 valores faltantes e imputar el resto.  
Como `hour.csv` no presenta valores faltantes, no se eliminan ni imputan variables. Aun así, se deja el procedimiento programado para documentar el cumplimiento.

In [ ]:
# Variables con más de 250 valores faltantes
variables_mas_250_faltantes = resumen_faltantes[
    resumen_faltantes["Valores faltantes"] > 250
]["Variable"].tolist()

print("Variables con más de 250 valores faltantes:")
print(variables_mas_250_faltantes)

# Copia para limpieza
df_limpio = df_eda.copy()

# Eliminación si existieran variables con más de 250 faltantes
if len(variables_mas_250_faltantes) > 0:
    df_limpio = df_limpio.drop(columns=variables_mas_250_faltantes)
    print("Se eliminaron variables con más de 250 valores faltantes.")
else:
    print("No se eliminaron variables porque ninguna supera 250 valores faltantes.")

# Imputación para el resto de variables con faltantes, si existieran
for columna in df_limpio.columns:
    if df_limpio[columna].isnull().sum() > 0:
        if pd.api.types.is_numeric_dtype(df_limpio[columna]):
            df_limpio[columna] = df_limpio[columna].fillna(df_limpio[columna].median())
        else:
            df_limpio[columna] = df_limpio[columna].fillna(df_limpio[columna].mode()[0])

print("Total de valores faltantes después del proceso:", df_limpio.isnull().sum().sum())

## 10. Visualizaciones exploratorias

Esta sección responde las preguntas del proyecto mediante gráficas de distribución, categorías y agrupaciones.

### 10.1. Distribución de la demanda total de bicicletas

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df_limpio["cnt"], bins=40, kde=True)
plt.title("Distribución de la demanda total de bicicletas por hora")
plt.xlabel("Cantidad total de bicicletas alquiladas")
plt.ylabel("Frecuencia")
plt.show()

## Grafica Hora del dia vs total de bicicletas alquiladas

In [ ]:
plt.figure(figsize=(14, 6))

sns.histplot(
    data=df_limpio,
    x="hr",
    weights="cnt",
    bins=24
)

plt.title("Total de bicicletas alquiladas por hora del día")
plt.xlabel("Hora del día")
plt.ylabel("Total de bicicletas alquiladas")
plt.xticks(range(0, 24))
plt.show()

Si quieres verlo como promedio por hora, que suele ser más fácil de interpretar

In [ ]:
plt.figure(figsize=(14, 6))

sns.barplot(
    data=df_limpio,
    x="hr",
    y="cnt",
    estimator="mean"
)

plt.title("Promedio de bicicletas alquiladas por hora del día")
plt.xlabel("Hora del día")
plt.ylabel("Promedio de bicicletas alquiladas")
plt.xticks(range(0, 24))
plt.show()

### 10.2. Demanda promedio por hora del día y tipo de día

In [ ]:
demanda_hora_tipo = df_limpio.groupby(["hr", "tipo_dia"], as_index=False)["cnt"].mean()

plt.figure(figsize=(14, 6))
sns.lineplot(data=demanda_hora_tipo, x="hr", y="cnt", hue="tipo_dia", marker="o")
plt.title("Demanda promedio por hora: día laboral vs fin de semana o feriado")
plt.xlabel("Hora del día")
plt.ylabel("Promedio de bicicletas alquiladas")
plt.xticks(range(0, 24))
plt.show()

### 10.3. Horas pico de uso por tipo de día

In [ ]:
horas_pico = (
    demanda_hora_tipo
    .sort_values(["tipo_dia", "cnt"], ascending=[True, False])
    .groupby("tipo_dia")
    .head(5)
    .reset_index(drop=True)
)

horas_pico

### 10.4. Comparación de usuarios casuales y registrados por hora

* La variable que habla sobre usuarios casuales serian la cantidad de usuarios casuales/no registrados que alquilaron bicicletas en esa hora.
* La variable que habla sobre usuarios registrados serian la cantidad de usuarios registrados/miembros del sistema que alquilaron bicicletas en esa hora.
* La variable cnt seria el total de bicicletas alquiladas. Segun la grafica, total de bicicletas alquiladas en esa hora.

In [ ]:
usuarios_hora = df_limpio.groupby("hr", as_index=False)[["casual", "registered"]].mean()

usuarios_hora_melt = usuarios_hora.melt(
    id_vars="hr",
    value_vars=["casual", "registered"],
    var_name="Tipo de usuario",
    value_name="Promedio de alquileres"
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=usuarios_hora_melt, x="hr", y="Promedio de alquileres", hue="Tipo de usuario", marker="o")
plt.title("Comportamiento horario de usuarios casuales y registrados")
plt.xlabel("Hora del día")
plt.ylabel("Promedio de alquileres")
plt.xticks(range(0, 24))
plt.show()

### 10.5. Usuarios casuales y registrados según tipo de día

In [ ]:
usuarios_hora_tipo = df_limpio.groupby(["hr", "tipo_dia"], as_index=False)[["casual", "registered"]].mean()

fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(data=usuarios_hora_tipo, x="hr", y="registered", hue="tipo_dia", marker="o", ax=ax)
plt.title("Usuarios registrados por hora y tipo de día")
plt.xlabel("Hora del día")
plt.ylabel("Promedio de usuarios registrados")
plt.xticks(range(0, 24))
plt.show()

fig, ax = plt.subplots(figsize=(14, 6))
sns.lineplot(data=usuarios_hora_tipo, x="hr", y="casual", hue="tipo_dia", marker="o", ax=ax)
plt.title("Usuarios casuales por hora y tipo de día")
plt.xlabel("Hora del día")
plt.ylabel("Promedio de usuarios casuales")
plt.xticks(range(0, 24))
plt.show()

### 10.6. Efecto del clima en la demanda por hora

In [ ]:
clima_hora = df_limpio.groupby(["hr", "weathersit_label"], as_index=False)["cnt"].mean()

plt.figure(figsize=(14, 6))
sns.lineplot(data=clima_hora, x="hr", y="cnt", hue="weathersit_label", marker="o")
plt.title("Demanda promedio por hora según condición climática")
plt.xlabel("Hora del día")
plt.ylabel("Promedio de bicicletas alquiladas")
plt.xticks(range(0, 24))
plt.legend(title="Condición climática", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

### 10.7. Mapa de calor de demanda por hora y día de la semana

In [ ]:
heatmap_hora_dia = df_limpio.pivot_table(
    values="cnt",
    index="weekday_label",
    columns="hr",
    aggfunc="mean"
)

orden_dias = ["Domingo", "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]
heatmap_hora_dia = heatmap_hora_dia.reindex(orden_dias)

plt.figure(figsize=(16, 6))
sns.heatmap(heatmap_hora_dia, cmap="YlOrRd", annot=False)
plt.title("Mapa de calor: demanda promedio por hora y día de la semana")
plt.xlabel("Hora del día")
plt.ylabel("Día de la semana")
plt.show()

### 10.8. Franjas horarias de demanda baja

Se identifican horas con demanda promedio baja. Esto ayuda a responder si existen franjas horarias con demanda consistentemente baja.

In [ ]:
demanda_promedio_hora = df_limpio.groupby("hr", as_index=False)["cnt"].mean()
umbral_bajo = demanda_promedio_hora["cnt"].quantile(0.25)

horas_baja_demanda = demanda_promedio_hora[demanda_promedio_hora["cnt"] <= umbral_bajo].sort_values("cnt")

print("Umbral de baja demanda:", round(umbral_bajo, 2))
horas_baja_demanda

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=demanda_promedio_hora, x="hr", y="cnt")
plt.axhline(umbral_bajo, linestyle="--", label="Umbral de baja demanda")
plt.title("Demanda promedio por hora con umbral de baja demanda")
plt.xlabel("Hora del día")
plt.ylabel("Promedio de bicicletas alquiladas")
plt.legend()
plt.show()

## 11. Análisis de correlación

Se presentan dos análisis de correlación:
1. Correlación de Pearson.
2. Correlación de Spearman.

Para evitar interpretaciones incorrectas, se debe recordar que `casual` y `registered` forman parte de `cnt`, por lo que tendrán correlaciones muy altas con la demanda total.

In [ ]:
variables_correlacion = [
    "season", "yr", "mnth", "hr", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed", "casual", "registered", "cnt"
]

corr_pearson = df_limpio[variables_correlacion].corr(method="pearson")

plt.figure(figsize=(14, 10))
sns.heatmap(corr_pearson, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación de Pearson")
plt.show()

In [ ]:
# Correlaciones de Pearson con la variable objetivo cnt
corr_cnt_pearson = corr_pearson["cnt"].sort_values(ascending=False)
corr_cnt_pearson

In [ ]:
corr_spearman = df_limpio[variables_correlacion].corr(method="spearman")

plt.figure(figsize=(14, 10))
sns.heatmap(corr_spearman, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación de Spearman")
plt.show()

In [ ]:
# Correlaciones de Spearman con la variable objetivo cnt
corr_cnt_spearman = corr_spearman["cnt"].sort_values(ascending=False)
corr_cnt_spearman

## 12. Preparación para modelos predictivos

Se entrenarán modelos para:

- **Clasificación:** predecir el nivel de demanda categórico: Baja, Media o Alta.
- **Regresión:** predecir la cantidad total de bicicletas alquiladas (`cnt`).

Para evitar fuga de información, no se usan `casual` ni `registered` como variables predictoras, porque son componentes directos de `cnt`.

In [ ]:
df_modelos = df_limpio.copy()

# Variable categórica para clasificación basada en terciles de cnt
df_modelos["nivel_demanda"] = pd.qcut(
    df_modelos["cnt"],
    q=3,
    labels=["Baja", "Media", "Alta"]
)

df_modelos["nivel_demanda"].value_counts()

In [ ]:
# Variables predictoras
features = [
    "season", "yr", "mnth", "hr", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed"
]

X = df_modelos[features]
y_clasificacion = df_modelos["nivel_demanda"]
y_regresion = df_modelos["cnt"]

print("Variables predictoras utilizadas:")
print(features)

## 13. Modelos de clasificación

Se entrenan dos algoritmos:

1. Random Forest Classifier.
2. K-Nearest Neighbors Classifier.

Se reporta Accuracy en entrenamiento y prueba, además de matrices de confusión.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X,
    y_clasificacion,
    test_size=0.2,
    random_state=42,
    stratify=y_clasificacion
)

print("Tamaño entrenamiento:", X_train_c.shape)
print("Tamaño prueba:", X_test_c.shape)

In [ ]:
modelos_clasificacion = {
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    "KNN Classifier": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ])
}

resultados_clasificacion = []

for nombre, modelo in modelos_clasificacion.items():
    modelo.fit(X_train_c, y_train_c)

    pred_train = modelo.predict(X_train_c)
    pred_test = modelo.predict(X_test_c)

    acc_train = accuracy_score(y_train_c, pred_train)
    acc_test = accuracy_score(y_test_c, pred_test)

    resultados_clasificacion.append({
        "Modelo": nombre,
        "Accuracy entrenamiento": acc_train,
        "Accuracy prueba": acc_test
    })

resultados_clasificacion_df = pd.DataFrame(resultados_clasificacion)
resultados_clasificacion_df

In [ ]:
mejor_modelo_clasificacion = resultados_clasificacion_df.loc[
    resultados_clasificacion_df["Accuracy prueba"].idxmax()
]

mejor_modelo_clasificacion

In [ ]:
# Matrices de confusión para entrenamiento y prueba

for nombre, modelo in modelos_clasificacion.items():
    print("\nModelo:", nombre)

    pred_train = modelo.predict(X_train_c)
    pred_test = modelo.predict(X_test_c)

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_train_c, pred_train, ax=ax, cmap="Blues")
    plt.title(f"Matriz de confusión - Entrenamiento - {nombre}")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test_c, pred_test, ax=ax, cmap="Blues")
    plt.title(f"Matriz de confusión - Prueba - {nombre}")
    plt.show()

## 14. Modelos de regresión

Se entrenan dos algoritmos:

1. Random Forest Regressor.
2. Support Vector Regressor.

Se reporta el coeficiente R² para entrenamiento y prueba. También se calcula RMSE y MAE como apoyo.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y_regresion,
    test_size=0.2,
    random_state=42
)

print("Tamaño entrenamiento:", X_train_r.shape)
print("Tamaño prueba:", X_test_r.shape)

In [ ]:
modelos_regresion = {
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    "SVR": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=100, gamma="scale", epsilon=0.1))
    ])
}

resultados_regresion = []

for nombre, modelo in modelos_regresion.items():
    modelo.fit(X_train_r, y_train_r)

    pred_train = modelo.predict(X_train_r)
    pred_test = modelo.predict(X_test_r)

    r2_train = r2_score(y_train_r, pred_train)
    r2_test = r2_score(y_test_r, pred_test)

    rmse_test = root_mean_squared_error(y_test_r, pred_test)
    mae_test = mean_absolute_error(y_test_r, pred_test)

    resultados_regresion.append({
        "Modelo": nombre,
        "R2 entrenamiento": r2_train,
        "R2 prueba": r2_test,
        "RMSE prueba": rmse_test,
        "MAE prueba": mae_test
    })

resultados_regresion_df = pd.DataFrame(resultados_regresion)
resultados_regresion_df

In [ ]:
mejor_modelo_regresion = resultados_regresion_df.loc[
    resultados_regresion_df["R2 prueba"].idxmax()
]

mejor_modelo_regresion

In [ ]:
# Comparación visual de valores reales vs predichos para cada modelo de regresión

for nombre, modelo in modelos_regresion.items():
    pred_test = modelo.predict(X_test_r)

    plt.figure(figsize=(7, 6))
    sns.scatterplot(x=y_test_r, y=pred_test, alpha=0.5)
    plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], linestyle="--")
    plt.title(f"Valores reales vs predichos - {nombre}")
    plt.xlabel("Valores reales de cnt")
    plt.ylabel("Valores predichos de cnt")
    plt.show()

## 15. Selección y almacenamiento del mejor modelo de regresión

El dashboard debe usar el modelo de regresión con mejor desempeño.  
Se selecciona el modelo con mayor R² en prueba y se guarda con `joblib`.

In [ ]:
mejor_modelo_nombre = resultados_regresion_df.sort_values(by="R2 prueba", ascending=False).iloc[0]["Modelo"]
mejor_modelo = modelos_regresion[mejor_modelo_nombre]

print("Mejor modelo de regresión:", mejor_modelo_nombre)

# Guardar el modelo y la lista de variables predictoras
Path("modelos").mkdir(exist_ok=True)

joblib.dump(mejor_modelo, "modelos/mejor_modelo_regresion_bike_sharing.pkl")
joblib.dump(features, "modelos/variables_modelo_bike_sharing.pkl")

print("Modelo guardado en: modelos/mejor_modelo_regresion_bike_sharing.pkl")
print("Variables guardadas en: modelos/variables_modelo_bike_sharing.pkl")

## 16. Desviacion estandar de la variable cnt

In [ ]:
# Variable a analizar
variable = "cnt"

# Cálculo de medidas estadísticas
media = df_limpio[variable].mean()
mediana = df_limpio[variable].median()
moda = df_limpio[variable].mode()[0]

q1 = df_limpio[variable].quantile(0.25)
q2 = df_limpio[variable].quantile(0.50)
q3 = df_limpio[variable].quantile(0.75)

desviacion = df_limpio[variable].std()
sesgo = df_limpio[variable].skew()

# Clasificación del sesgo
if sesgo > 0.5:
    tipo_sesgo = "Sesgo positivo"
elif sesgo < -0.5:
    tipo_sesgo = "Sesgo negativo"
else:
    tipo_sesgo = "Distribución aproximadamente simétrica"

# Mostrar resultados
print("Media:", round(media, 2))
print("Mediana:", round(mediana, 2))
print("Moda:", round(moda, 2))
print("Q1:", round(q1, 2))
print("Q2 / Mediana:", round(q2, 2))
print("Q3:", round(q3, 2))
print("Desviación estándar:", round(desviacion, 2))
print("Sesgo:", round(sesgo, 2))
print("Tipo de sesgo:", tipo_sesgo)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Histograma con curva suave
sns.histplot(df_limpio[variable], bins=40, kde=True, ax=ax)

# Líneas de medidas estadísticas
ax.axvline(media, linestyle="--", linewidth=2, label=f"Media: {media:.2f}")
ax.axvline(mediana, linestyle="-", linewidth=2, label=f"Mediana: {mediana:.2f}")
ax.axvline(moda, linestyle=":", linewidth=2, label=f"Moda: {moda:.2f}")

# Cuartiles
ax.axvline(q1, linestyle="--", linewidth=1.5, label=f"Q1: {q1:.2f}")
ax.axvline(q2, linestyle="-", linewidth=1.5, label=f"Q2: {q2:.2f}")
ax.axvline(q3, linestyle="--", linewidth=1.5, label=f"Q3: {q3:.2f}")

# Rango de una desviación estándar alrededor de la media
ax.axvspan(media - desviacion, media + desviacion, alpha=0.15, label="Media ± 1 desviación estándar")

plt.title(f"Distribución de la demanda total de bicicletas\n{tipo_sesgo}")
plt.xlabel("Cantidad total de bicicletas alquiladas por hora")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

Esta gráfica muestra la distribución de la variable `cnt`, que representa la cantidad total de bicicletas alquiladas por hora. Se incluyen medidas estadísticas como la media, mediana, moda, cuartiles y desviación estándar para analizar la concentración y dispersión de los datos.

El sesgo permite identificar si la distribución está balanceada o inclinada hacia algún extremo. Si la media es mayor que la mediana, generalmente existe sesgo positivo; si la media es menor que la mediana, puede existir sesgo negativo. En este caso, la clasificación del sesgo se obtuvo mediante el coeficiente de asimetría de la variable.

## 17. Grafica StripPlot + BoxPlot

### BoxPlot + StripPlot por tipo de día

In [ ]:
# Variable categórica y variable numérica
grupo = "tipo_dia"
variable = "cnt"

# Se toma una muestra para que los puntos del StripPlot no saturen la gráfica
df_muestra = df_limpio[[grupo, variable]].dropna().sample(
    n=min(3000, len(df_limpio)),
    random_state=42
)

plt.figure(figsize=(12, 6))

# Boxplot con todos los datos
sns.boxplot(
    data=df_limpio,
    x=grupo,
    y=variable,
    showfliers=True
)

# Stripplot con una muestra de puntos
sns.stripplot(
    data=df_muestra,
    x=grupo,
    y=variable,
    jitter=True,
    alpha=0.35,
    size=3
)

plt.title("Distribución de la demanda de bicicletas por tipo de día")
plt.xlabel("Tipo de día")
plt.ylabel("Cantidad total de bicicletas alquiladas")
plt.show()

### BoxPlot + StripPlot por condición climática

In [ ]:
grupo = "weathersit_label"
variable = "cnt"

df_muestra = df_limpio[[grupo, variable]].dropna().sample(
    n=min(3000, len(df_limpio)),
    random_state=42
)

plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df_limpio,
    x=grupo,
    y=variable,
    showfliers=True
)

sns.stripplot(
    data=df_muestra,
    x=grupo,
    y=variable,
    jitter=True,
    alpha=0.35,
    size=3
)

plt.title("Distribución de la demanda de bicicletas según condición climática")
plt.xlabel("Condición climática")
plt.ylabel("Cantidad total de bicicletas alquiladas")
plt.xticks(rotation=20)
plt.show()

La combinación de BoxPlot y StripPlot permite analizar la distribución de la demanda de bicicletas por grupo. El BoxPlot muestra la mediana, los cuartiles y posibles valores atípicos, mientras que el StripPlot permite observar la dispersión real de los registros. Esta visualización ayuda a identificar si existen diferencias relevantes en la demanda según el grupo analizado, además de mostrar valores que se encuentran fuera del rango típico de comportamiento.

## 18. Conclusiones preliminares del Entregable 1

Completar después de ejecutar el notebook:

- El dataset `hour.csv` contiene **17379 observaciones** y **17 variables**.
- No se encontraron valores faltantes en las variables analizadas, por lo que no fue necesario eliminar ni imputar datos.
- Las horas con mayor demanda en días laborales fueron: **17, 18, 8, 19, 16**.
- Las horas con mayor demanda en fines de semana o feriados fueron: **13, 12, 14, 15, 16**.
- Los usuarios registrados presentan un patrón de uso más marcado en horas relacionadas con traslados diarios, mientras que los usuarios casuales tienden a concentrarse en horarios distintos.
- Las condiciones climáticas muestran diferencias en la demanda, especialmente cuando se comparan días despejados con condiciones de lluvia o nieve ligera.
- En clasificación, el modelo con mejor desempeño fue **Random Forest Classifier**.
- En regresión, el modelo con mejor desempeño fue **Random Forest Classifier**, por lo que será utilizado posteriormente en el dashboard.

## 19. Resumen de cumplimiento del Entregable 1

| Requisito del Entregable 1 | Estado |
|---|---|
| Análisis preliminar del dataset | Cumplido |
| Medidas de tendencia central | Cumplido |
| Análisis de distribuciones | Cumplido |
| Visualización de datos | Cumplido |
| Gráficas de categorías / agrupadas | Cumplido |
| Análisis de valores faltantes con missingno | Cumplido |
| Análisis de valores faltantes con UpSetPlot | Cumplido/documentado |
| Eliminación o imputación de datos faltantes | Cumplido/documentado |
| Al menos 2 análisis de correlación | Cumplido |
| 2 modelos de clasificación | Cumplido |
| Accuracy entrenamiento y prueba | Cumplido |
| Matriz de confusión entrenamiento y prueba | Cumplido |
| 2 modelos de regresión | Cumplido |
| R² entrenamiento y prueba | Cumplido |
| Selección de mejor modelo para dashboard | Cumplido |